# Entrenamiento en Colab (flujo híbrido)

Colab **entrena**; tu máquina local **evalúa y visualiza**. Checkpoints en Drive (persisten), logs en disco local de Colab y sincronizados a Drive cada pocos minutos.

Antes: *Entorno de ejecución → Cambiar tipo de entorno → GPU*. Sube `mario_rl_dia1.zip` a `MyDrive/mario/` (o usa un repo Git).

In [1]:
# 1) GPU
!nvidia-smi
import torch; print("CUDA:", torch.cuda.is_available())

Sat Oct 24 21:22:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2) Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# 3) Código del proyecto desde GitHub
REPO_URL = "https://github.com/David-AGL/mario-bros-reinforcement-learning.git"
BRANCH = "main"   # cambia si trabajas en otra rama

import os
if not os.path.exists("/content/mario/.git"):
    !git clone -b {BRANCH} {REPO_URL} /content/mario
else:
    !git -C /content/mario pull origin {BRANCH}
%cd /content/mario
!git log --oneline -1

From https://github.com/David-AGL/mario-bros-reinforcement-learning
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/mario
0b54b00 (HEAD -> main, origin/main, origin/HEAD) conection to repository


In [4]:
# 4) Dependencias (mismas versiones que requirements.txt, sin tocar torch de Colab)
!pip install -q "numpy<2" "gym==0.26.2" "nes-py==8.2.1" "gym-super-mario-bros==7.4.0" "gymnasium==0.29.1" "stable-baselines3==2.3.2" opencv-python-headless psutil tensorboard
# Si pip avisa de conflictos con NumPy o pide reiniciar: Entorno de ejecución -> Reiniciar sesión, y vuelve a ejecutar desde la celda 2 (omite esta).

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.2 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompa

In [4]:
# 5) Smoke test del entorno + FPS en Colab (compáralo con tu portátil)
!python -W ignore scripts/smoke_test_env.py

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
SuperMarioBros-1-1-v0: OK | episodios=5 max_x=1398 FPS(agent steps)=93
SuperMarioBros-2-1-v0: OK | episodios=18 max_x=899 FPS(agent steps)=92
SuperMarioBros-3-1-v0: OK | episodios=15 max_x=780 FPS(agent steps)=95
SuperMarioBros-1-2-v0: OK | episodios=3 max_x=1139 FPS(agent steps)=92
No-ops reproducibles con la misma semilla: [1, 1, 1]
SMOKE TEST OK


## Configuración de la corrida
`TOTAL_STEPS` debe ser **el mismo para DQN y PPO**; fíjalo tras medir FPS.

In [5]:
SEED = 1
TOTAL_STEPS = 50_000        # smoke test del día 2; luego el presupuesto real
RUN_NAME = f"dqn_s{SEED}"   # usa otro nombre para pruebas, p. ej. dqn_smoke

DRIVE = "/content/drive/MyDrive/mario"
MODELS_ROOT = f"{DRIVE}/models"          # checkpoints: persisten en Drive
LOGS_ROOT = "/content/runs"              # logs: disco local (rápido)
LOGS_DRIVE = f"{DRIVE}/runs"             # copia de seguridad de logs

# En Colab (~12 GB RAM gratis) conviene reducir el buffer. Ajusta tras ver el dry-run.
EXTRA = "--optimize-memory-usage"

In [7]:
!pip uninstall -y -q tensorflow tensorflow-cpu jax jaxlib
!python -c "from stable_baselines3 import DQN; print('SB3 OK')"

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
SB3 OK


In [8]:
# 6) Dry-run: configuración + memoria estimada (no entrena)
!python -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA} --dry-run

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Config: {
  "level": "SuperMarioBros-1-1-v0",
  "noop_max": 30,
  "seed": 1,
  "total_steps": 50000,
  "learning_rate": 0.0001,
  "buffer_size": 100000,
  "learning_starts": 20000,
  "batch_size": 32,
  "gamma": 0.99,
  "train_freq": 4,
  "gradient_steps": 1,
  "target_update_interval": 10000,
  "exploration_fraction": 0.2,
  "exploration_initial_eps": 1.0,
  "exploration_final_eps": 0.05,
  "max_grad_norm": 10.0,
  "optimize_memory_usage": true,
  "checkpoint_freq": 100000,
  "save_replay_buffe

In [9]:
# 7) Sincronizar logs a Drive en segundo plano cada 5 min
import threading, time, subprocess, os
os.makedirs(LOGS_ROOT, exist_ok=True); os.makedirs(LOGS_DRIVE, exist_ok=True)
_stop = False
def _sync():
    while not _stop:
        subprocess.run(["rsync", "-a", f"{LOGS_ROOT}/", f"{LOGS_DRIVE}/"])
        time.sleep(300)
threading.Thread(target=_sync, daemon=True).start()

In [10]:
# 8) ENTRENAR (si es una reanudación, usa la celda 9)
!python -W ignore -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Config: {
  "level": "SuperMarioBros-1-1-v0",
  "noop_max": 30,
  "seed": 1,
  "total_steps": 50000,
  "learning_rate": 0.0001,
  "buffer_size": 100000,
  "learning_starts": 20000,
  "batch_size": 32,
  "gamma": 0.99,
  "train_freq": 4,
  "gradient_steps": 1,
  "target_update_interval": 10000,
  "exploration_fraction": 0.2,
  "exploration_initial_eps": 1.0,
  "exploration_final_eps": 0.05,
  "max_grad_norm": 10.0,
  "optimize_memory_usage": true,
  "checkpoint_freq": 100000,
  "save_replay_buffe

## Reanudar tras una caída
Busca el último checkpoint en Drive y reanuda con el **mismo** `RUN_NAME` y `TOTAL_STEPS`.
Sin replay buffer guardado, el buffer se vuelve a llenar (queda registrado en la salida).

In [11]:
# 9) Reanudar
import glob, re
ckpts = glob.glob(f"{MODELS_ROOT}/dqn/{RUN_NAME}/dqn_s{SEED}_*_steps.zip")
last = max(ckpts, key=lambda p: int(re.search(r"_(\d+)_steps", p).group(1)))
print("Reanudando desde:", last)
!python -W ignore -m dqn.train --seed {SEED} --total-steps {TOTAL_STEPS} --run-name {RUN_NAME} \
    --models-root {MODELS_ROOT} --logs-root {LOGS_ROOT} {EXTRA} --resume {last}

ValueError: max() iterable argument is empty

In [12]:
!ls -lh {MODELS_ROOT}/dqn/{RUN_NAME}

total 27M
-rw------- 1 root root 1.1K Oct 24 21:27 config.json
-rw------- 1 root root  27M Oct 24 21:37 dqn_s1_final.zip


In [13]:
# 10) Al terminar: sincronizar logs y listar lo que hay en Drive
_stop = True
!rsync -a {LOGS_ROOT}/ {LOGS_DRIVE}/
!ls -lh {MODELS_ROOT}/dqn/{RUN_NAME}

total 27M
-rw------- 1 root root 1.1K Oct 24 21:27 config.json
-rw------- 1 root root  27M Oct 24 21:37 dqn_s1_final.zip


## En tu máquina local
Descarga de Drive `mario/models/` y `mario/runs/` a las carpetas `models/` y `runs/` del repo. Luego:
`tensorboard --logdir runs/` y evalúa con `evaluate.py` (solo inferencia, funciona en CPU).